# Death Knight Lab · 02
## Let the decks play

**Choose Run → Run All Cells. No extra installation needed.**

This notebook plays complete matches between **random players** using **33 implemented cards**. It shows a progress bar, saves completed matches, and gives you a readable replay. It does **not** train a model yet.

The 117-card catalog from notebook 01 is larger than the playable pool. We generate fresh decks from the supported subset so no card effect is silently skipped. This is an experimental rules implementation, not full Standard or a live-client-verified simulator.

In [ ]:
from pathlib import Path
import sys, importlib

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "lab.py").exists() and (p / "engine").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook inside your death-knight-lab project folder.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Reload the supporting code when you rerun this notebook after an update.
import engine.cards, engine.game, engine.experiment, engine.notebook
for module in (engine.cards, engine.game, engine.experiment, engine.notebook):
    importlib.reload(module)
from engine.cards import supported_pool, coverage
from engine.experiment import run_experiment, replay_game, match_log
from engine.notebook import Progress, show_summary
from lab import describe
pool = supported_pool()
print(f"Ready: {len(pool)} implemented cards. Results save inside {root / 'runs'}")

### Your settings

Start with 100 games. Later, try 1,000. The game count must be even because each pairing plays twice with the starting player reversed.

`DECKS_PER_RUNE_PROFILE = 2` creates 20 random decks. Each deck obeys rune and copy limits.

**Same settings = resume the same run.** Change `RANDOM_SEED` for fresh decks and games. Use the square Stop button to interrupt; completed games are saved automatically.

In [ ]:
NUMBER_OF_GAMES = 100
DECKS_PER_RUNE_PROFILE = 2
RANDOM_SEED = 42
RUN_RULE_CHECKS = True

### Check the rules first

These checks cover card effects, legal actions, hidden information, and reproducibility. A failing check stops the notebook before the batch begins. Passing them is useful evidence, but not proof of perfect agreement with the live game.

In [ ]:
if RUN_RULE_CHECKS:
    import subprocess
    check = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"],
        cwd=root, capture_output=True, text=True,
    )
    print(check.stdout + check.stderr)
    if check.returncode:
        raise RuntimeError("Rule checks failed. No batch was started.")
else:
    print("Rule checks skipped by your setting.")

### Play the matches

Both players choose random legal moves. The progress bar measures completed games—not how much the agent has learned. A match may end by lethal damage, fatigue, or the turn limit. Errors stop the run and preserve completed results.

In [ ]:
progress = Progress()
experiment = run_experiment(
    games=NUMBER_OF_GAMES,
    decks_per_profile=DECKS_PER_RUNE_PROFILE,
    seed=RANDOM_SEED,
    output_dir=root / "runs",
    progress=progress,
)
print("Saved experiment:", experiment["directory"])

### What happened?

This is a random-play baseline, **not a ranking of decks or runes**. The limited pool and weak players favor some strategies over others. The starting player is balanced within each pair of matches.

In [ ]:
show_summary(experiment)

### Inspect the first matchup

Here are the two decks from the first game. We save both lists in the replay so the experiment can be reproduced. During play, neither agent can see the other player's hidden cards.

In [ ]:
sample = experiment["sample"]
for player, deck, runes in zip(("A", "B"), sample["decks"], sample["profiles"]):
    print(f"\nPLAYER {player} — Blood / Frost / Unholy: {runes}")
    print(describe(deck, pool))

### Watch the turns

Change `FIRST_TURN` and `LAST_TURN` and run this cell again to read a different part of the match. Each turn header shows both heroes' health and corpse counts at the start of the turn. Damage, healing, attacks, and deaths appear below it.

In [ ]:
FIRST_TURN = 1
LAST_TURN = 10
print(match_log(sample, first_turn=FIRST_TURN, last_turn=LAST_TURN))

### Verify the replay

Replaying the recorded actions with the original seed must reproduce the exact public event log and result.

In [ ]:
replayed = replay_game(sample)
outcome = "Draw" if replayed.winner is None else f"Player {'A' if replayed.winner == 0 else 'B'} wins"
print(f"Replay verified: {outcome}; {replayed.turn} turns; {replayed.end_reason}.")

### Which cards are playable?

Every supported card has an explicit implementation. All other cards in the catalog are rejected. Discover and unrestricted random generation are excluded for now because they would require a much larger pool of implemented effects.

In [ ]:
for card in sorted(pool, key=lambda c: (c["cardClass"], c["cost"], c["name"])):
    print(f"{card['cardClass']:12} | {card['cost']:2} mana | {card['name']}")
print(f"\n{len(pool)} supported; {sum(not c['supported'] for c in coverage())} catalog cards excluded.")

### Next: learn to play

The next notebook will replace a random player with a trainable policy, starting from random weights. It will learn from game outcomes and be evaluated against a fixed random baseline and earlier checkpoints. After playing skill improves, we can evolve the decks through card swaps.

This notebook saves no trained weights because no training has occurred. Our immediate milestone is **reproducible games with tested rules**. Before using results to judge actual Standard decks, we must expand card coverage, verify interactions against the live game, and introduce other classes.